In [1]:
import json
import numpy as np
from pathlib import Path

from model_ranking import (
    to_target_transfer_correlations,
    correlation_table,
    load_transfer_metric_results,
    match_model_names,
    avg_correlation_table,
    avg_correlation_to_latex,
    avg_correlation_to_latex_transposed,
)

INFO: P [MainThread] 2025-11-05 11:10:35,294 plantseg - Logger configured at initialisation. PlantSeg logger name: plantseg


/g/kreshuk/talks/pytorch-3dunet/pytorch3dunet/unet3d/utils.py:17: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/g/kreshuk/talks/miniforge3/envs/model-rank-local2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
performance_path = "/g/kreshuk/talks/consistency_results/patch_segmentation/mitochondria/transfer_results/consistency/self_supervised/transfer_self_supervised_Finetuned_Def_performance_scores.json"
with open(performance_path, "r") as f:
    performance_scores = json.load(f)["performance_scores"]


# Gauss EI (Finetuned)

In [3]:
base_path = "/g/kreshuk/talks/consistency_results/patch_segmentation/mitochondria/transfer_results/consistency/self_supervised/CMB"
selected_augmentations = ["a015-02"]

for aug in selected_augmentations:
    file_name = f"transfer_self_supervised_Gauss_{aug}_CMB_05f_05b_EI_scores.json"
    consistency_path = Path(base_path) / file_name
    results = load_transfer_metric_results(consistency_path)
    consistency_scores = results["transfer_scores"]
    targets = list(consistency_scores.keys())
    consistency_scores = match_model_names(
        performance_scores=performance_scores,
        transfer_scores=consistency_scores,
    )
    KT_scores, SP_scores, PE_scores = to_target_transfer_correlations(
        targets=targets,
        transfer_metric_per_target=consistency_scores,
        performance_per_target=performance_scores,
        )
    df_gauss_EI = correlation_table(KT_scores, SP_scores, PE_scores, targets=targets)
    df_avg_gauss_EI = avg_correlation_table(df_gauss_EI)
    print(df_avg_gauss_EI)

Loaded Transfer metric results from: /g/kreshuk/talks/consistency_results/patch_segmentation/mitochondria/transfer_results/consistency/self_supervised/CMB/transfer_self_supervised_Gauss_a015-02_CMB_05f_05b_EI_scores.json
      KT_avg    KT_std  SR_avg    SR_std  PR_avg    PR_std
Task                                                      
Mito    0.55  0.248328   0.675  0.303809    0.65  0.260128


# DO EI (finetuned)

In [4]:
base_path = "/g/kreshuk/talks/consistency_results/patch_segmentation/mitochondria/transfer_results/feature_perturbation_consistency/self_supervised/EI_consistency/CMB"
selected_augmentations = ["a002"]

for aug in selected_augmentations:
    file_name = f"transfer_DO_{aug}_CMB_05f_05b_EI_scores.json"
    consistency_path = Path(base_path) / file_name
    results = load_transfer_metric_results(consistency_path)
    consistency_scores = results["transfer_scores"]
    targets = list(consistency_scores.keys())
    consistency_scores = match_model_names(
        performance_scores=performance_scores,
        transfer_scores=consistency_scores,
    )
    KT_scores, SP_scores, PE_scores = to_target_transfer_correlations(
        targets=targets,
        transfer_metric_per_target=consistency_scores,
        performance_per_target=performance_scores,
        )
    df_DO_EI = correlation_table(KT_scores, SP_scores, PE_scores, targets=targets)
    df_avg_DO_EI = avg_correlation_table(df_DO_EI)
    print(df_avg_DO_EI)

Loaded Transfer metric results from: /g/kreshuk/talks/consistency_results/patch_segmentation/mitochondria/transfer_results/feature_perturbation_consistency/self_supervised/EI_consistency/CMB/transfer_DO_a002_CMB_05f_05b_EI_scores.json
      KT_avg    KT_std  SR_avg    SR_std  PR_avg    PR_std
Task                                                      
Mito    0.46  0.231948  0.6075  0.271093  0.6225  0.347982


# Latex avg table

In [5]:
avg_dfs = [
    df_avg_gauss_EI,
    df_avg_DO_EI,
]
metric_names = [
    "CTE-EI",
    "CTE-EI",
]
metric_specs = [
    "Gauss",
    "DropOut",
]

latex_table = avg_correlation_to_latex(
    df_avg_list=avg_dfs,
    metric_name_list=metric_names,
    metric_spec_list=metric_specs,
)
print(latex_table)

\begin{table}[htbp]
    \centering
    \setlength{\tabcolsep}{3pt}
    \begin{tabular}{cc|ccc}
    \hline
    \multirow{2}{*}{Metric} & {} & \multicolumn{3}{c}{Mito} \\
 &  & K$\tau$ & S$\rho$ & P$r$ \\
\hline
\multirow{2}{*}{\begin{tabular}{@{}c@{}} CTE-EI \\ (Gauss)\end{tabular}} & \textbf{Avg.} & 0.55 & 0.68 & 0.65 \\
 & \textbf{std.} & ±0.25 & ±0.30 & ±0.26 \\
\hline
\multirow{2}{*}{\begin{tabular}{@{}c@{}} CTE-EI \\ (DropOut)\end{tabular}} & \textbf{Avg.} & 0.46 & 0.61 & 0.62 \\
 & \textbf{std.} & ±0.23 & ±0.27 & ±0.35 \\
\hline
\end{tabular}
\caption{Correlation scores for multiple metrics}
\label{tab:multiple_metrics}
\end{table}


# Test Transposed Table Format

In [6]:
# Test the transposed format with the same data
latex_table_transposed = avg_correlation_to_latex_transposed(
    df_avg_list=avg_dfs,
    metric_name_list=metric_names,
    metric_spec_list=metric_specs,
)
print(latex_table_transposed)

\begin{table*}[htbp]
    \centering
    \setlength{\tabcolsep}{1.5pt}
    \begin{tabular}{cc|cc|cc}
    \hline
    \multirow{2}{*}{Task} &  & \multicolumn{2}{c|}{\begin{tabular}{@{}c@{}} CTE-EI \\ (Gauss)\end{tabular}} & \multicolumn{2}{c}{\begin{tabular}{@{}c@{}} CTE-EI \\ (DropOut)\end{tabular}} \\
 &  & \textbf{Avg.} & \textbf{std.} & \textbf{Avg.} & \textbf{std.} \\
\hline
\multirow{3}{*}{\rotatebox[origin=c]{90}{Mito}} & K$\tau$ & 0.55 & ±0.2 & 0.46 & ±0.2 \\
 & S$\rho$ & 0.68 & ±0.3 & 0.61 & ±0.3 \\
 & P$r$ & 0.65 & ±0.3 & 0.62 & ±0.3 \\
\hline
\end{tabular}
\caption{Correlation scores for multiple metrics (transposed)}
\label{tab:multiple_metrics_transposed}
\end{table*}


# Without VNC

In [13]:
df_DO_EI

kt  kt pval  s rho  s rho pval    pr  pr pval
Task targets                                                 
Mito EPFL     0.60     0.01   0.75        0.01  0.91     0.00
     Hmito    0.42     0.08   0.63        0.06  0.60     0.05
     Rmito    0.67     0.00   0.83        0.01  0.84     0.00
     VNC      0.15     0.55   0.22        0.51  0.14     0.66

In [16]:
df_DO_EI_no_VNC = df_DO_EI[df_DO_EI.index.get_level_values('targets').isin(['EPFL', 'Hmito', 'Rmito'])]
df_avg_DO_EI_no_VNC = avg_correlation_table(df_DO_EI_no_VNC)
print(df_avg_DO_EI_no_VNC)

        KT_avg   KT_std    SR_avg    SR_std    PR_avg    PR_std
Task                                                           
Mito  0.563333  0.12897  0.736667  0.100664  0.783333  0.162583


In [ ]:
df_gauss_EI_no_VNC = df_gauss_EI[df_gauss_EI.index.get_level_values('targets').isin(['EPFL', 'Hmito', 'Rmito'])]
df_avg_gauss_EI_no_VNC = avg_correlation_table(df_gauss_EI_no_VNC)
print(df_avg_gauss_EI_no_VNC)

                kt  kt pval  s rho  s rho pval    pr  pr pval
Task targets                                                 
Mito EPFL     0.67     0.00   0.85        0.00  0.89     0.00
     Hmito    0.71     0.00   0.82        0.01  0.72     0.01
     Rmito    0.64     0.01   0.81        0.00  0.71     0.01
        KT_avg    KT_std    SR_avg    SR_std    PR_avg   PR_std
Task                                                           
Mito  0.673333  0.035119  0.826667  0.020817  0.773333  0.10116


## Latex table (NO VNC)

In [19]:
avg_dfs = [
    df_avg_gauss_EI_no_VNC,
    df_avg_DO_EI_no_VNC,
]
metric_names = [
    "CTE-EI",
    "CTE-EI",
]
metric_specs = [
    "Gauss",
    "DropOut",
]

latex_table = avg_correlation_to_latex(
    df_avg_list=avg_dfs,
    metric_name_list=metric_names,
    metric_spec_list=metric_specs,
)
print(latex_table)

\begin{table}[htbp]
    \centering
    \setlength{\tabcolsep}{3pt}
    \begin{tabular}{cc|ccc}
    \hline
    \multirow{2}{*}{Metric} & {} & \multicolumn{3}{c}{Mito} \\
 &  & K$\tau$ & S$\rho$ & P$r$ \\
\hline
\multirow{2}{*}{\begin{tabular}{@{}c@{}} CTE-EI \\ (Gauss)\end{tabular}} & \textbf{Avg.} & 0.67 & 0.83 & 0.77 \\
 & \textbf{std.} & ±0.04 & ±0.02 & ±0.10 \\
\hline
\multirow{2}{*}{\begin{tabular}{@{}c@{}} CTE-EI \\ (DropOut)\end{tabular}} & \textbf{Avg.} & 0.56 & 0.74 & 0.78 \\
 & \textbf{std.} & ±0.13 & ±0.10 & ±0.16 \\
\hline
\end{tabular}
\caption{Correlation scores for multiple metrics}
\label{tab:multiple_metrics}
\end{table}
